In [2]:
# ============================================================
# BranchyViT — STANDALONE MEASUREMENT SCRIPT
#
# Loads the trained BranchyViT checkpoint produced by the fixed
# training script (branchynet_vit_cifar10_fixed.py / document 7)
# and measures BOTH:
#   (a) standard accuracy/FLOPs/size/latency — final exit only
#   (b) true adaptive-computation behavior — threshold sweep over
#       adaptive_forward(), including per-exit usage fractions and
#       isolated single-image adaptive latency
#
# WHY THIS FILE EXISTS (vs. reusing document 6 directly):
#   document 6 ("baseline" evaluator) instantiates a PLAIN timm ViT
#   via build_model(), not BranchyViT. The trained checkpoint here
#   has branch1/branch2 heads and a restructured set of submodules
#   (patch_embed, cls_token, blocks, norm, head as direct attributes)
#   that a plain timm.create_model(...) state_dict will NOT match.
#   Pointing document 6 at this .pth file will raise a state_dict
#   key-mismatch error. This script reconstructs the correct class.
#
#   document 6 also never calls adaptive_forward() — it only times
#   the standard 3-exit forward pass, so none of the "adaptive
#   computation" story (the whole point of BranchyNet) gets measured.
#   This script restores that sweep using the same methodology as
#   the fixed training script (CUDA events / sync+perf_counter).
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm
import time, os, json, tempfile, random
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 10
IMG_SIZE    = 32
PATCH_SIZE  = 4
TIMM_BASE   = 'vit_small_patch16_224'
BATCH_SIZE  = 128

# FIX: must exactly match the training script's SAVE_PATH, including
# the .pth extension. document 6 had "__5__branchynet_vit_cifar10_fixed"
# (no extension) while the training script saves
# "__5__branchynet_vit_cifar10_fixed.pth" — those are different files.
SAVE_PATH = "__5__branchynet_vit_cifar10.pth"

EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
BRANCH_WEIGHTS  = [0.2, 0.3, 0.5]   # for reporting only — not used at eval time

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

# ── UTILITY FUNCTIONS (same methodology as the fixed training script) ──
def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)
    return round(size, 4)

def compute_flops(model, device, input_size=(1, 3, IMG_SIZE, IMG_SIZE)):
    """Hook-based FLOPs counter — same blind spot as thop (no attention
    matmul accounting), but consistent across ResNet/ViT comparisons."""
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        N, C_out, H_out, W_out = out.shape
        C_in = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) \
                  else (module.kernel_size, module.kernel_size)
        total_flops[0] += 2 * N * C_out * H_out * W_out * (C_in // module.groups) * kH * kW

    def linear_hook(module, inp, out):
        N = inp[0].shape[0]
        total_flops[0] += 2 * N * module.in_features * module.out_features

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    with torch.no_grad():
        # Use the standard forward (returns 3 logits tuple) for FLOPs counting
        m(torch.randn(*input_size, device=device))
    for h in hooks:
        h.remove()
    return round(total_flops[0] / 1e9, 6)


# ── AUXILIARY BRANCH (must match training script exactly) ─────
class EarlyExitBranch(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.branch = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        cls = x[:, 0]
        return self.branch(cls)


# ── BRANCHYVIT (must match training script exactly so state_dict
#    keys line up — copied structurally from the fixed training script) ──
class BranchyViT(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, img_size: int = IMG_SIZE,
                 patch_size: int = PATCH_SIZE):
        super().__init__()

        backbone = timm.create_model(
            TIMM_BASE, pretrained=False, num_classes=num_classes,
            img_size=img_size, patch_size=patch_size,
        )

        self.patch_embed = backbone.patch_embed
        self.cls_token   = backbone.cls_token
        self.pos_embed   = backbone.pos_embed
        self.pos_drop    = backbone.pos_drop
        self.blocks      = backbone.blocks
        self.norm        = backbone.norm
        self.head        = backbone.head

        hidden_dim = backbone.embed_dim
        self.num_classes = num_classes
        self.img_size    = img_size
        self.patch_size  = patch_size

        self.branch1 = EarlyExitBranch(hidden_dim, num_classes)
        self.branch2 = EarlyExitBranch(hidden_dim, num_classes)

        self.exit1_block_idx = 3
        self.exit2_block_idx = 7

    def forward(self, x):
        """Full forward — all three exits. Used for standard-accuracy eval
        and for FLOPs/latency measurement of the non-adaptive path."""
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_drop(x + self.pos_embed)

        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if i == self.exit1_block_idx:
                logits1 = self.branch1(x)
            elif i == self.exit2_block_idx:
                logits2 = self.branch2(x)

        x       = self.norm(x)
        logits3 = self.head(x[:, 0])

        return logits1, logits2, logits3

    @torch.no_grad()
    def adaptive_forward(self, imgs, threshold: float = 0.8):
        """TRUE per-sample early exit — identical logic to the training
        script. Confident samples are sliced out and never execute
        deeper transformer blocks."""
        B      = imgs.shape[0]
        device = imgs.device

        final_logits     = torch.zeros(B, self.num_classes, device=device)
        exit_idx         = torch.full((B,), 2, dtype=torch.long, device=device)
        remaining_global  = torch.arange(B, device=device)

        h   = self.patch_embed(imgs)
        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)
        h   = self.pos_drop(h + self.pos_embed)

        for i in range(self.exit1_block_idx + 1):
            h = self.blocks[i](h)

        logits1 = self.branch1(h)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values
        done1   = conf1 >= threshold
        if done1.any():
            g_idx = remaining_global[done1]
            final_logits[g_idx] = logits1[done1]
            exit_idx[g_idx]     = 0

        keep = ~done1
        if not keep.any():
            return final_logits, exit_idx

        h                = h[keep]
        remaining_global = remaining_global[keep]

        for i in range(self.exit1_block_idx + 1, self.exit2_block_idx + 1):
            h = self.blocks[i](h)

        logits2 = self.branch2(h)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values
        done2   = conf2 >= threshold
        if done2.any():
            g_idx = remaining_global[done2]
            final_logits[g_idx] = logits2[done2]
            exit_idx[g_idx]     = 1

        keep = ~done2
        if not keep.any():
            return final_logits, exit_idx

        h                = h[keep]
        remaining_global = remaining_global[keep]

        for i in range(self.exit2_block_idx + 1, len(self.blocks)):
            h = self.blocks[i](h)

        h       = self.norm(h)
        logits3 = self.head(h[:, 0])

        final_logits[remaining_global] = logits3
        return final_logits, exit_idx


# ── LOAD TRAINED BRANCHYVIT ────────────────────────────────────
if not os.path.exists(SAVE_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found at '{SAVE_PATH}'. Check that the training "
        f"script's SAVE_PATH matches this exactly (including extension)."
    )

model = BranchyViT(num_classes=NUM_CLASSES, img_size=IMG_SIZE, patch_size=PATCH_SIZE)
state_dict = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(state_dict, strict=True)   # raises loudly on any key mismatch
model = model.to(DEVICE).eval()
print(f"✓ BranchyViT weights loaded from {SAVE_PATH}")

total_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {total_params:,}")

# ── DATA ─────────────────────────────────────────────────────
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_set    = torchvision.datasets.CIFAR10(root='../data', train=False,
                                            download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE,
                                           shuffle=False, num_workers=0)

# ── STANDARD METRICS (final exit only) ─────────────────────────
print("\n" + "="*60)
print("STANDARD EVALUATION — final exit only")
print("="*60)

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1        = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc:.4f}")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")

# ── ADAPTIVE COMPUTATION EVALUATION (missing from document 6) ──
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)
print(f"  Thresholds tested: {EXIT_THRESHOLDS}")

adaptive_results = []
for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []
    t_start = time.time()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc_t  = accuracy_score(labels_arr, preds_arr)
    prec_t = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec_t  = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1_t   = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc_t,  6),
        "precision"  : round(prec_t, 6),
        "recall"     : round(rec_t,  6),
        "f1"         : round(f1_t,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc_t:.4f} | F1={f1_t:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

# ── FLOPs & DISK SIZE ──────────────────────────────────────────
flops_G = compute_flops(model, DEVICE)
size_mb = disk_size_mb(model)

# # ── CPU INFERENCE TIMING (standard forward, all 3 exits) ───────
# print("\n⏱  Measuring CPU inference times ...")
# model_cpu = BranchyViT(num_classes=NUM_CLASSES, img_size=IMG_SIZE, patch_size=PATCH_SIZE)
# model_cpu.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
# model_cpu = model_cpu.eval()

# dummy_single_cpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
# dummy_batch_cpu  = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE)

# with torch.no_grad():
#     for _ in range(10):
#         model_cpu(dummy_single_cpu)

# times_cpu_single = []
# with torch.no_grad():
#     for _ in range(100):
#         t0 = time.perf_counter()
#         model_cpu(dummy_single_cpu)
#         times_cpu_single.append(time.perf_counter() - t0)
# inf_ms_single_cpu = np.mean(times_cpu_single) * 1000

# with torch.no_grad():
#     for _ in range(5):
#         model_cpu(dummy_batch_cpu)

# times_cpu_batch = []
# with torch.no_grad():
#     for _ in range(20):
#         t0 = time.perf_counter()
#         model_cpu(dummy_batch_cpu)
#         times_cpu_batch.append(time.perf_counter() - t0)
# inf_ms_batch128_cpu     = np.mean(times_cpu_batch) * 1000
# inf_ms_per_img_cpu      = inf_ms_batch128_cpu / BATCH_SIZE
# throughput_imgs_sec_cpu = BATCH_SIZE / (inf_ms_batch128_cpu / 1000)

# del model_cpu, dummy_single_cpu, dummy_batch_cpu
# print("✓ CPU timing done")

# ── GPU INFERENCE TIMING (standard forward, all 3 exits) ───────
use_cuda = DEVICE.type == "cuda"
print("⏱  Measuring GPU inference times ...")
dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

with torch.no_grad():
    for _ in range(50):
        model(dummy_single_gpu)
if use_cuda:
    torch.cuda.synchronize()

times_gpu_single = []
with torch.no_grad():
    for _ in range(500):
        if use_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(dummy_single_gpu)
        if use_cuda:
            torch.cuda.synchronize()
        times_gpu_single.append(time.perf_counter() - t0)
inf_ms_single_gpu = np.mean(times_gpu_single) * 1000

dummy_batch_gpu = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
if use_cuda:
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)
    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch_gpu)
    torch.cuda.synchronize()

    batch_times_gpu = []
    with torch.no_grad():
        for _ in range(100):
            start_ev.record()
            model(dummy_batch_gpu)
            end_ev.record()
            torch.cuda.synchronize()
            batch_times_gpu.append(start_ev.elapsed_time(end_ev))
    inf_ms_batch128_gpu = np.mean(batch_times_gpu)
else:
    with torch.no_grad():
        for _ in range(5):
            model(dummy_batch_gpu)
    batch_times_gpu = []
    with torch.no_grad():
        for _ in range(20):
            t0 = time.perf_counter()
            model(dummy_batch_gpu)
            batch_times_gpu.append((time.perf_counter() - t0) * 1000)
    inf_ms_batch128_gpu = np.mean(batch_times_gpu)

inf_ms_per_img_gpu      = inf_ms_batch128_gpu / BATCH_SIZE
throughput_imgs_sec_gpu = BATCH_SIZE / (inf_ms_batch128_gpu / 1000)
print("✓ GPU timing done")

# ── ADAPTIVE LATENCY @ τ=0.8 (single image, isolated) ───────────
dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(dummy_single_gpu, threshold=0.8)
    if use_cuda:
        torch.cuda.synchronize()
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt   = torch.cuda.Event(enable_timing=True)
        start_evt.record()
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
        end_evt.record()
        torch.cuda.synchronize()
        inf_ms_adapt_tau08 = start_evt.elapsed_time(end_evt) / 50
    else:
        t0 = time.perf_counter()
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
        inf_ms_adapt_tau08 = (time.perf_counter() - t0) / 50 * 1000

# ── PRINT SUMMARY ────────────────────────────────────────────
print(f"\n{'='*60}")
print("BRANCHYVIT METRICS SUMMARY")
print(f"{'='*60}")
print(f"  Accuracy (final exit) : {acc:.4f}")
print(f"  Parameters            : {total_params:,}")
print(f"  Model size             : {size_mb:.4f} MB")
print(f"  FLOPs (full, all exits): {flops_G:.4f} G")
# print(f"  --- Inference (CPU, full model) ---")
# print(f"  Single image       : {inf_ms_single_cpu:.3f} ms")
# print(f"  Batch-128          : {inf_ms_batch128_cpu:.2f} ms")
# print(f"  Throughput         : {throughput_imgs_sec_cpu:.1f} imgs/sec")
print(f"  --- Inference (GPU, full model) ---")
print(f"  Single image       : {inf_ms_single_gpu:.3f} ms")
print(f"  Batch-128          : {inf_ms_batch128_gpu:.2f} ms")
print(f"  Throughput         : {throughput_imgs_sec_gpu:.1f} imgs/sec")
print(f"  --- Adaptive (τ=0.8, single image) ---")
print(f"  Latency            : {inf_ms_adapt_tau08:.3f} ms")
print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
print(f"    Accuracy   : {best_adaptive['accuracy']:.4f}")
print(f"    Exit1      : {best_adaptive['frac_exit1']:.1%}")
print(f"    Exit2      : {best_adaptive['frac_exit2']:.1%}")
print(f"    Exit3      : {best_adaptive['frac_exit3']:.1%}")

# ── SAVE METRICS JSON ────────────────────────────────────────
branchyvit_metrics = {
    "model"     : "branchyvit_small_patch4_32",
    "method"    : "early_exit",
    "variant"   : "BranchyViT_Small",
    "dataset"   : "CIFAR-10",
    "accuracy"  : round(acc, 6),
    "precision" : round(precision, 6),
    "recall"    : round(recall, 6),
    "f1"        : round(f1, 6),
    "params"    : total_params,
    "size_mb"   : size_mb,
    "flops_G"   : flops_G,
    "input_resolution": IMG_SIZE,
    "patch_size": PATCH_SIZE,
    "num_patches": (IMG_SIZE // PATCH_SIZE) ** 2,
    "inference_ms": {
        # "cpu": {
        #     "single_img_ms"      : round(inf_ms_single_cpu, 4),
        #     "batch128_ms"        : round(inf_ms_batch128_cpu, 4),
        #     "per_img_ms"         : round(inf_ms_per_img_cpu, 4),
        #     "throughput_imgs_sec": round(throughput_imgs_sec_cpu, 1),
        #     "timing_method"      : "time.perf_counter()",
        # },
        "gpu": {
            "single_img_ms"      : round(inf_ms_single_gpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_gpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_gpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_gpu, 1),
            "timing_method"      : "CUDA events + torch.cuda.synchronize()" if use_cuda else "time.perf_counter()",
        },
    },
    "num_exits"                   : 3,
    "exit_positions"              : [
        f"after block {model.exit1_block_idx} of 11 (~33% depth)",
        f"after block {model.exit2_block_idx} of 11 (~67% depth)",
        "final head (full depth)",
    ],
    "branch_weights"              : BRANCH_WEIGHTS,
    "exit_thresholds_tested"      : EXIT_THRESHOLDS,
    "inference_ms_adaptive_tau08" : round(inf_ms_adapt_tau08, 4),
    "adaptive_threshold_results"  : adaptive_results,
    "best_adaptive_result"        : best_adaptive,
    "saved_model_path"            : os.path.abspath(SAVE_PATH),
}

output_json = "__5__branchynet_vit_standalone_metrics.json"
with open(output_json, "w") as f:
    json.dump(branchyvit_metrics, f, indent=2)
print(f"\n✓ Metrics saved → {output_json}")

Using device: cuda
✓ BranchyViT weights loaded from __5__branchynet_vit_cifar10.pth
  Parameters: 21,445,022


e:\baseline_ViT\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



STANDARD EVALUATION — final exit only
  Accuracy          : 0.9715
  Precision (macro) : 0.9715
  Recall    (macro) : 0.9715
  F1-score  (macro) : 0.9715

ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep
  Thresholds tested: [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
  τ=0.50 | Acc=0.9314 | F1=0.9313 | Exit1=92.5% Exit2=6.6% Exit3=1.0% | Time=0.2826ms/sample
  τ=0.60 | Acc=0.9433 | F1=0.9432 | Exit1=87.8% Exit2=10.2% Exit3=2.0% | Time=0.2912ms/sample
  τ=0.70 | Acc=0.9544 | F1=0.9543 | Exit1=82.4% Exit2=14.0% Exit3=3.6% | Time=0.2945ms/sample
  τ=0.80 | Acc=0.9629 | F1=0.9628 | Exit1=74.1% Exit2=18.8% Exit3=7.1% | Time=0.3068ms/sample
  τ=0.90 | Acc=0.9693 | F1=0.9693 | Exit1=54.4% Exit2=27.8% Exit3=17.7% | Time=0.3417ms/sample
  τ=0.95 | Acc=0.9712 | F1=0.9712 | Exit1=13.4% Exit2=0.6% Exit3=86.0% | Time=0.4949ms/sample
⏱  Measuring GPU inference times ...
✓ GPU timing done

BRANCHYVIT METRICS SUMMARY
  Accuracy (final exit) : 0.9715
  Parameters            : 21,445,022
  Model size            

In [3]:
# ============================================================
# BranchyViT — STANDALONE MEASUREMENT SCRIPT — FIXED
#
# Same two bugs fixed here as in branchynet_vit_cifar10_fixed.py:
#
#   1. compute_flops()'s linear_hook used N = inp[0].shape[0], which
#      is only correct for 2D (B, features) tensors. ViT's nn.Linear
#      layers operate on 3D (B, num_tokens, D) tensors, so this
#      undercounted FLOPs by ~65x (num_tokens = 64 patches + 1 CLS).
#      Fixed to use N = in_t.numel() // in_t.shape[-1], which counts
#      every leading dimension, not just batch.
#
#   2. The τ=0.8 adaptive single-image latency benchmark used
#      torch.randn() noise. Confidence-based early exit can't trigger
#      on noise (near-uniform softmax never clears 0.8), so this was
#      silently measuring ~full-depth latency regardless of the
#      threshold sweep showing most real images exit early. Fixed to
#      use a real test image instead.
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm
import time, os, json, tempfile, random
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 10
IMG_SIZE    = 32
PATCH_SIZE  = 4
TIMM_BASE   = 'vit_small_patch16_224'
BATCH_SIZE  = 128

SAVE_PATH = "__5__branchynet_vit_cifar10.pth"

EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
BRANCH_WEIGHTS  = [0.2, 0.3, 0.5]   # for reporting only — not used at eval time

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

# ── UTILITY FUNCTIONS (same methodology as the fixed training script) ──
def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)
    return round(size, 4)

def compute_flops(model, device, input_size=(1, 3, IMG_SIZE, IMG_SIZE)):
    """Hook-based FLOPs counter — same blind spot as thop (no attention
    matmul accounting), but consistent across ResNet/ViT comparisons."""
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        N, C_out, H_out, W_out = out.shape
        C_in = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) \
                  else (module.kernel_size, module.kernel_size)
        total_flops[0] += 2 * N * C_out * H_out * W_out * (C_in // module.groups) * kH * kW

    def linear_hook(module, inp, out):
        # FIX: nn.Linear broadcasts over every leading dimension, not
        # just the batch dim. For a 2D ResNet-style tensor (B, features)
        # this is a no-op change (N = B either way). For a 3D ViT tensor
        # (B, num_tokens, D), the old version used N = B and dropped the
        # num_tokens factor, undercounting FLOPs by ~65x.
        in_t = inp[0]
        N = in_t.numel() // in_t.shape[-1]
        total_flops[0] += 2 * N * module.in_features * module.out_features

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    with torch.no_grad():
        # Use the standard forward (returns 3 logits tuple) for FLOPs counting
        m(torch.randn(*input_size, device=device))
    for h in hooks:
        h.remove()
    return round(total_flops[0] / 1e9, 6)


# ── AUXILIARY BRANCH (must match training script exactly) ─────
class EarlyExitBranch(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.branch = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        cls = x[:, 0]
        return self.branch(cls)


# ── BRANCHYVIT (must match training script exactly so state_dict
#    keys line up — copied structurally from the fixed training script) ──
class BranchyViT(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, img_size: int = IMG_SIZE,
                 patch_size: int = PATCH_SIZE):
        super().__init__()

        backbone = timm.create_model(
            TIMM_BASE, pretrained=False, num_classes=num_classes,
            img_size=img_size, patch_size=patch_size,
        )

        self.patch_embed = backbone.patch_embed
        self.cls_token   = backbone.cls_token
        self.pos_embed   = backbone.pos_embed
        self.pos_drop    = backbone.pos_drop
        self.blocks      = backbone.blocks
        self.norm        = backbone.norm
        self.head        = backbone.head

        hidden_dim = backbone.embed_dim
        self.num_classes = num_classes
        self.img_size    = img_size
        self.patch_size  = patch_size

        self.branch1 = EarlyExitBranch(hidden_dim, num_classes)
        self.branch2 = EarlyExitBranch(hidden_dim, num_classes)

        self.exit1_block_idx = 3
        self.exit2_block_idx = 7

    def forward(self, x):
        """Full forward — all three exits. Used for standard-accuracy eval
        and for FLOPs/latency measurement of the non-adaptive path."""
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_drop(x + self.pos_embed)

        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if i == self.exit1_block_idx:
                logits1 = self.branch1(x)
            elif i == self.exit2_block_idx:
                logits2 = self.branch2(x)

        x       = self.norm(x)
        logits3 = self.head(x[:, 0])

        return logits1, logits2, logits3

    @torch.no_grad()
    def adaptive_forward(self, imgs, threshold: float = 0.8):
        """TRUE per-sample early exit — identical logic to the training
        script. Confident samples are sliced out and never execute
        deeper transformer blocks."""
        B      = imgs.shape[0]
        device = imgs.device

        final_logits     = torch.zeros(B, self.num_classes, device=device)
        exit_idx         = torch.full((B,), 2, dtype=torch.long, device=device)
        remaining_global  = torch.arange(B, device=device)

        h   = self.patch_embed(imgs)
        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)
        h   = self.pos_drop(h + self.pos_embed)

        for i in range(self.exit1_block_idx + 1):
            h = self.blocks[i](h)

        logits1 = self.branch1(h)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values
        done1   = conf1 >= threshold
        if done1.any():
            g_idx = remaining_global[done1]
            final_logits[g_idx] = logits1[done1]
            exit_idx[g_idx]     = 0

        keep = ~done1
        if not keep.any():
            return final_logits, exit_idx

        h                = h[keep]
        remaining_global = remaining_global[keep]

        for i in range(self.exit1_block_idx + 1, self.exit2_block_idx + 1):
            h = self.blocks[i](h)

        logits2 = self.branch2(h)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values
        done2   = conf2 >= threshold
        if done2.any():
            g_idx = remaining_global[done2]
            final_logits[g_idx] = logits2[done2]
            exit_idx[g_idx]     = 1

        keep = ~done2
        if not keep.any():
            return final_logits, exit_idx

        h                = h[keep]
        remaining_global = remaining_global[keep]

        for i in range(self.exit2_block_idx + 1, len(self.blocks)):
            h = self.blocks[i](h)

        h       = self.norm(h)
        logits3 = self.head(h[:, 0])

        final_logits[remaining_global] = logits3
        return final_logits, exit_idx


# ── LOAD TRAINED BRANCHYVIT ────────────────────────────────────
if not os.path.exists(SAVE_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found at '{SAVE_PATH}'. Check that the training "
        f"script's SAVE_PATH matches this exactly (including extension)."
    )

model = BranchyViT(num_classes=NUM_CLASSES, img_size=IMG_SIZE, patch_size=PATCH_SIZE)
state_dict = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(state_dict, strict=True)   # raises loudly on any key mismatch
model = model.to(DEVICE).eval()
print(f"✓ BranchyViT weights loaded from {SAVE_PATH}")

total_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {total_params:,}")

# ── DATA ─────────────────────────────────────────────────────
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_set    = torchvision.datasets.CIFAR10(root='../data', train=False,
                                            download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE,
                                           shuffle=False, num_workers=0)

# ── STANDARD METRICS (final exit only) ─────────────────────────
print("\n" + "="*60)
print("STANDARD EVALUATION — final exit only")
print("="*60)

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1        = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc:.4f}")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")

# ── ADAPTIVE COMPUTATION EVALUATION ─────────────────────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)
print(f"  Thresholds tested: {EXIT_THRESHOLDS}")

adaptive_results = []
for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []
    t_start = time.time()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc_t  = accuracy_score(labels_arr, preds_arr)
    prec_t = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec_t  = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1_t   = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc_t,  6),
        "precision"  : round(prec_t, 6),
        "recall"     : round(rec_t,  6),
        "f1"         : round(f1_t,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc_t:.4f} | F1={f1_t:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

# ── FLOPs & DISK SIZE ──────────────────────────────────────────
flops_G = compute_flops(model, DEVICE)
size_mb = disk_size_mb(model)

# ── GPU INFERENCE TIMING (standard forward, all 3 exits) ───────
use_cuda = DEVICE.type == "cuda"
print("⏱  Measuring GPU inference times ...")
dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

with torch.no_grad():
    for _ in range(50):
        model(dummy_single_gpu)
if use_cuda:
    torch.cuda.synchronize()

times_gpu_single = []
with torch.no_grad():
    for _ in range(500):
        if use_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(dummy_single_gpu)
        if use_cuda:
            torch.cuda.synchronize()
        times_gpu_single.append(time.perf_counter() - t0)
inf_ms_single_gpu = np.mean(times_gpu_single) * 1000

dummy_batch_gpu = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
if use_cuda:
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)
    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch_gpu)
    torch.cuda.synchronize()

    batch_times_gpu = []
    with torch.no_grad():
        for _ in range(100):
            start_ev.record()
            model(dummy_batch_gpu)
            end_ev.record()
            torch.cuda.synchronize()
            batch_times_gpu.append(start_ev.elapsed_time(end_ev))
    inf_ms_batch128_gpu = np.mean(batch_times_gpu)
else:
    with torch.no_grad():
        for _ in range(5):
            model(dummy_batch_gpu)
    batch_times_gpu = []
    with torch.no_grad():
        for _ in range(20):
            t0 = time.perf_counter()
            model(dummy_batch_gpu)
            batch_times_gpu.append((time.perf_counter() - t0) * 1000)
    inf_ms_batch128_gpu = np.mean(batch_times_gpu)

inf_ms_per_img_gpu      = inf_ms_batch128_gpu / BATCH_SIZE
throughput_imgs_sec_gpu = BATCH_SIZE / (inf_ms_batch128_gpu / 1000)
print("✓ GPU timing done")

# ── ADAPTIVE LATENCY @ τ=0.8 (single image, isolated) ───────────
# FIX: uses a REAL test image, not torch.randn() noise — see header
# comment. Random noise produces a near-uniform softmax that almost
# never clears a 0.8 confidence threshold, so the old version of this
# block was silently measuring close to full-depth latency every time.
real_batch, _ = next(iter(test_loader))
real_single   = real_batch[:1].to(DEVICE)

with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(real_single, threshold=0.8)
    if use_cuda:
        torch.cuda.synchronize()
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt   = torch.cuda.Event(enable_timing=True)
        start_evt.record()
        for _ in range(50):
            model.adaptive_forward(real_single, threshold=0.8)
        end_evt.record()
        torch.cuda.synchronize()
        inf_ms_adapt_tau08 = start_evt.elapsed_time(end_evt) / 50
    else:
        t0 = time.perf_counter()
        for _ in range(50):
            model.adaptive_forward(real_single, threshold=0.8)
        inf_ms_adapt_tau08 = (time.perf_counter() - t0) / 50 * 1000

# ── PRINT SUMMARY ────────────────────────────────────────────
print(f"\n{'='*60}")
print("BRANCHYVIT METRICS SUMMARY")
print(f"{'='*60}")
print(f"  Accuracy (final exit)  : {acc:.4f}")
print(f"  Parameters             : {total_params:,}")
print(f"  Model size              : {size_mb:.4f} MB")
print(f"  FLOPs (full, all exits) : {flops_G:.4f} G")
print(f"  --- Inference (GPU, full model) ---")
print(f"  Single image       : {inf_ms_single_gpu:.3f} ms")
print(f"  Batch-128          : {inf_ms_batch128_gpu:.2f} ms")
print(f"  Throughput         : {throughput_imgs_sec_gpu:.1f} imgs/sec")
print(f"  --- Adaptive (τ=0.8, single REAL image) ---")
print(f"  Latency            : {inf_ms_adapt_tau08:.3f} ms")
print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
print(f"    Accuracy   : {best_adaptive['accuracy']:.4f}")
print(f"    Exit1      : {best_adaptive['frac_exit1']:.1%}")
print(f"    Exit2      : {best_adaptive['frac_exit2']:.1%}")
print(f"    Exit3      : {best_adaptive['frac_exit3']:.1%}")

# ── SAVE METRICS JSON ────────────────────────────────────────
branchyvit_metrics = {
    "model"     : "branchyvit_small_patch4_32",
    "method"    : "early_exit",
    "variant"   : "BranchyViT_Small",
    "dataset"   : "CIFAR-10",
    "accuracy"  : round(acc, 6),
    "precision" : round(precision, 6),
    "recall"    : round(recall, 6),
    "f1"        : round(f1, 6),
    "params"    : total_params,
    "size_mb"   : size_mb,
    "flops_G"   : flops_G,
    "input_resolution": IMG_SIZE,
    "patch_size": PATCH_SIZE,
    "num_patches": (IMG_SIZE // PATCH_SIZE) ** 2,
    "inference_ms": {
        "gpu": {
            "single_img_ms"      : round(inf_ms_single_gpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_gpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_gpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_gpu, 1),
            "timing_method"      : "CUDA events + torch.cuda.synchronize()" if use_cuda else "time.perf_counter()",
        },
    },
    "num_exits"                   : 3,
    "exit_positions"              : [
        f"after block {model.exit1_block_idx} of 11 (~33% depth)",
        f"after block {model.exit2_block_idx} of 11 (~67% depth)",
        "final head (full depth)",
    ],
    "branch_weights"              : BRANCH_WEIGHTS,
    "exit_thresholds_tested"      : EXIT_THRESHOLDS,
    "inference_ms_adaptive_tau08" : round(inf_ms_adapt_tau08, 4),
    "adaptive_threshold_results"  : adaptive_results,
    "best_adaptive_result"        : best_adaptive,
    "saved_model_path"            : os.path.abspath(SAVE_PATH),
}

output_json = "__5__branchynet_vit_standalone_metrics.json"
with open(output_json, "w") as f:
    json.dump(branchyvit_metrics, f, indent=2)
print(f"\n✓ Metrics saved → {output_json}")

Using device: cuda
✓ BranchyViT weights loaded from __5__branchynet_vit_cifar10.pth
  Parameters: 21,445,022


e:\baseline_ViT\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



STANDARD EVALUATION — final exit only
  Accuracy          : 0.9715
  Precision (macro) : 0.9715
  Recall    (macro) : 0.9715
  F1-score  (macro) : 0.9715

ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep
  Thresholds tested: [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
  τ=0.50 | Acc=0.9314 | F1=0.9313 | Exit1=92.5% Exit2=6.6% Exit3=1.0% | Time=0.2833ms/sample
  τ=0.60 | Acc=0.9433 | F1=0.9432 | Exit1=87.8% Exit2=10.2% Exit3=2.0% | Time=0.2880ms/sample
  τ=0.70 | Acc=0.9544 | F1=0.9543 | Exit1=82.4% Exit2=14.0% Exit3=3.6% | Time=0.2967ms/sample
  τ=0.80 | Acc=0.9629 | F1=0.9628 | Exit1=74.1% Exit2=18.8% Exit3=7.1% | Time=0.3083ms/sample
  τ=0.90 | Acc=0.9693 | F1=0.9693 | Exit1=54.4% Exit2=27.8% Exit3=17.7% | Time=0.3461ms/sample
  τ=0.95 | Acc=0.9712 | F1=0.9712 | Exit1=13.4% Exit2=0.6% Exit3=86.0% | Time=0.4976ms/sample
⏱  Measuring GPU inference times ...
✓ GPU timing done

BRANCHYVIT METRICS SUMMARY
  Accuracy (final exit)  : 0.9715
  Parameters             : 21,445,022
  Model size          